# 04 — Data visualizations

Charts are based on available CSV columns and can be regenerated with `python src/analyze_data.py`. The static figures use Matplotlib and Seaborn; the final example is interactive Plotly.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.data_loader import load_datasets, PROCESSED_DATA_DIR
datasets = load_datasets(PROCESSED_DATA_DIR) or load_datasets(ROOT / 'data' / 'raw', require_all=True)
movies = datasets['movies']
chart_dir = ROOT / 'outputs' / 'charts'
chart_dir.mkdir(parents=True, exist_ok=True)
sns.set_theme(style='whitegrid')

## Release trends
This chart counts rows by parsed year. It helps reveal how the dataset is distributed over time; it should not be interpreted as historical global production totals.

In [ ]:
release_year = pd.to_datetime(movies['release_date'], errors='coerce').dt.year.dropna()
year_counts = release_year.value_counts().sort_index()
fig, ax = plt.subplots(figsize=(12, 5))
sns.lineplot(x=year_counts.index, y=year_counts.values, marker='o', ax=ax)
ax.set(title='Movies by Release Year', xlabel='Release year', ylabel='Number of movies')
fig.tight_layout()
fig.savefig(chart_dir / 'movies_by_year.png', dpi=150)
plt.show()

## Rating distribution
The histogram shows the distribution of available average votes. Ratings summarize the supplied records and can be influenced by vote count and selection bias.

In [ ]:
ratings = pd.to_numeric(movies['vote_average'], errors='coerce').dropna()
fig, ax = plt.subplots(figsize=(9, 5))
sns.histplot(ratings, bins=20, kde=True, ax=ax)
ax.set(title='Distribution of Movie Ratings', xlabel='Average vote', ylabel='Number of movies')
fig.tight_layout()
fig.savefig(chart_dir / 'rating_distribution.png', dpi=150)
plt.show()

## Genre distribution (interactive)
Hover over bars to inspect counts. The bridge table is grouped by genre name and distinct movie ID.

In [ ]:
genre_data = datasets['movie_genres'].merge(datasets['genres'], on='genre_id', how='left')
genre_counts = (genre_data.groupby('genre_name', dropna=False)['movie_id'].nunique().rename('movie_count').reset_index().sort_values('movie_count', ascending=False))
fig = px.bar(genre_counts, x='genre_name', y='movie_count', title='Movies by Genre', labels={'genre_name': 'Genre', 'movie_count': 'Unique movies'})
fig.update_layout(xaxis_tickangle=-35)
fig.show()
fig.write_html(chart_dir / 'genre_distribution_interactive.html', include_plotlyjs='cdn')

## Budget and revenue relationship
Zero values are treated as unavailable for this comparison. A scatter plot can show association but does not establish causality.

In [ ]:
financial = movies[['title', 'budget', 'revenue']].copy()
financial[['budget', 'revenue']] = financial[['budget', 'revenue']].apply(pd.to_numeric, errors='coerce')
financial = financial.dropna(subset=['budget', 'revenue'])
financial = financial[(financial['budget'] > 0) & (financial['revenue'] > 0)]
fig = px.scatter(financial, x='budget', y='revenue', hover_name='title', log_x=True, log_y=True, title='Budget and Revenue (positive values, log scales)')
fig.show()
print(f'Included {len(financial):,} movies with positive budget and revenue.')